# A CNN on one GH200

A small demonstration of the pattern the course notebooks follow on the CSCS JupyterHub for Santis: probe what the container already has, install the little that is missing, put data on scratch, and run the model on the GPU. It trains a convolutional network on FashionMNIST, 60,000 images of clothing in ten classes.

Run it in a session spawned with one GPU, as the Setup page of the course website describes, top to bottom with Shift+Enter.

In [ ]:
# Environment: what the image has, what we add, and which device we get
import importlib, os, subprocess, sys, sysconfig, time

def probe(*names):
    """Print the version of each importable package, or 'missing'."""
    for name in names:
        try:
            print("   %-13s %s" % (name, getattr(importlib.import_module(name), "__version__", "present")))
        except ImportError:
            print("   %-13s missing" % name)

print("in the image, before installing anything:")
probe("torch", "torchvision", "numpy", "pandas", "matplotlib", "sklearn", "transformers")

# Install only what is missing. The container is writable, so a plain pip install works and lands inside it;
# the container is rebuilt at every session, so that install is gone next time. Where site-packages is not
# writable, pip needs --user, and the package lands in ~/.local, which persists.
_user_site = [] if os.access(sysconfig.get_paths()["purelib"], os.W_OK) else ["--user"]
for module, package in (("torchvision", "torchvision"), ("matplotlib", "matplotlib")):
    try:
        importlib.import_module(module)
    except ImportError:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_user_site, package], check=False)

import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("\ntorch %s · CUDA %s · torchvision %s" % (torch.__version__, torch.version.cuda, torchvision.__version__))
if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print("GPU: %s · %.0f GB" % (gpu.name, gpu.total_memory / 1e9))
else:
    print("/!\\ NO GPU: this session was spawned without one. Spawn again with GPUs = 1. The rest runs on the CPU, slowly.")

### Adding a package the image lacks

The probe above reported scikit-learn missing. Anything the image lacks is added the same way the cell above added its own two packages: install only if the import fails, with the package name pip knows (`scikit-learn`) mapped to the module name Python imports (`sklearn`). The install lands inside the container and is gone at the next session, which is fine for a ten-second install; add `--user` to the pip call to keep it in `~/.local` instead. No kernel restart is needed for a new package.

In [ ]:
# Install scikit-learn only if it is missing, then probe again
for module, package in (("sklearn", "scikit-learn"),):
    try:
        importlib.import_module(module)
    except ImportError:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_user_site, package], check=False)

print("after installing:")
probe("sklearn")

from sklearn.metrics import confusion_matrix      # and it imports: used after training to read the model's mistakes

### Where files go

The home directory is 50 GB and meant for code. Datasets, checkpoints and caches belong on `$SCRATCH`, which is large and deletes files unused for 30 days. The container already sets `HF_HOME` there, so Hugging Face downloads land on scratch without any code; for torchvision we choose the folder ourselves.

In [ ]:
# Data: FashionMNIST, stored on scratch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DATA_DIR = os.path.join(os.environ.get("SCRATCH", "."), "fashion_mnist")
train_set = datasets.FashionMNIST(DATA_DIR, train=True,  download=True, transform=transforms.ToTensor())
test_set  = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_set, batch_size=256,  shuffle=True,  num_workers=4, pin_memory=DEVICE == "cuda")
test_loader  = DataLoader(test_set,  batch_size=1024, shuffle=False, num_workers=4, pin_memory=DEVICE == "cuda")
print("%d training images · %d test images · stored in %s" % (len(train_set), len(test_set), DATA_DIR))

fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, (x, y) in zip(axes, train_set):
    ax.imshow(x[0], cmap="gray")
    ax.set_title(train_set.classes[y], fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# The model, and where it lives
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)     # 28×28 → 28×28, pooled to 14×14
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)    # 14×14 → 14×14, pooled to 7×7
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)                 # ten classes

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.relu(self.fc1(x.flatten(1)))
        return self.fc2(x)

model = CNN().to(DEVICE)                                # every parameter tensor is copied to the GPU here, once
print("%s parameters · weights on %s" % (format(sum(p.numel() for p in model.parameters()), ","),
                                          next(model.parameters()).device))

In [ ]:
# Training on the GPU
def accuracy(net, loader):
    net.eval(); correct = 0
    with torch.no_grad():
        for x, y in loader:
            correct += (net(x.to(DEVICE)).argmax(1) == y.to(DEVICE)).sum().item()
    net.train()
    return correct / len(loader.dataset)

optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3)
for epoch in range(1, 3):
    start, running = time.perf_counter(), 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)   # each batch is moved to the GPU
        loss = F.cross_entropy(model(x), y)
        optimiser.zero_grad(); loss.backward(); optimiser.step()
        running += loss.item() * len(x)
    line = "epoch %d · loss %.3f · test accuracy %.1f%% · %.1f s" % (
        epoch, running / len(train_set), 100 * accuracy(model, test_loader), time.perf_counter() - start)
    if DEVICE == "cuda":
        line += " · peak GPU memory %.2f GB" % (torch.cuda.max_memory_allocated() / 1e9)
    print(line)

# What the trained model confuses, with scikit-learn: one row per true class, one column per prediction
import numpy as np
model.eval(); preds, trues = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds.append(model(x.to(DEVICE)).argmax(1).cpu()); trues.append(y)
cm = confusion_matrix(torch.cat(trues), torch.cat(preds))
off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print("most confused pair: %d test images of %s predicted as %s" % (off[i, j], train_set.classes[i], train_set.classes[j]))

### What this showed

- **Probe, then install.** The image ships PyTorch with CUDA and, as the first cell printed, a good part of the scientific stack. What a notebook needs beyond that it installs itself in a few seconds, and the check-then-install loop makes the same cell correct on a laptop, on Colab and here.
- **Data on scratch.** `DATA_DIR` came from `$SCRATCH`; the home directory stays for code.
- **Two `.to(DEVICE)` calls.** One for the model, once; one per batch. That is all it takes for the whole computation to run on the GH200; the training cell prints its peak memory there.
- When you are done, stop the server (File → Hub Control Panel → Stop My Server) so the GPU goes back to the pool.